

# Curso de Minería de Datos – Módulo 1
## Fundamentos y Exploración de Datos

**Objetivo:** ilustrar, paso a paso, cómo detectar y mitigar problemas comunes de calidad en un dataset y prepararlo para el modelado.

### Notebook práctico integrador
*Universidad Tecnológica de Pereira*

## Mapa CRISP‑DM → Secciones del notebook  
1. **Comprensión de los datos** → Carga + Diagnóstico de calidad  
2. **Preparación de los datos** → Imputación, outliers, codificación, escalado  
3. **Modelado** → *Fuera de alcance en este módulo*  
4. **Evaluación** → Análisis crítico / reflexiones  


In [ ]:
# Librerías base
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Librerías extra (instalar si es necesario)
# !pip install ydata-profiling category_encoders scikit-learn scipy statsmodels


## 1. Comprensión de los datos – Carga y vistazo inicial

In [ ]:
# Usaremos el dataset Titanic de seaborn
df = sns.load_dataset('titanic')
print(f"Dimensiones iniciales: {df.shape}")
df.head()

In [ ]:
# Información básica
df.info()
display(df.describe(include='all').T)

### Diagnóstico de calidad de datos

In [ ]:
# Visualizar valores faltantes
plt.figure(figsize=(10,4))
sns.heatmap(df.isna(), cbar=False)
plt.title('Mapa de valores faltantes')
plt.show()

In [ ]:
# Contar registros duplicados
n_dup = df.duplicated().sum()
print(f'Registros duplicados: {n_dup}')

# Eliminar duplicados si existen
if n_dup > 0:
    df = df.drop_duplicates()
    print(f"Dimensiones después de eliminar duplicados: {df.shape}")

## 2. Preparación – Imputación avanzada de valores faltantes

### Paso didáctico: imputación tradicional (mediana) → comparación → imputación k-NN

In [ ]:
# --- Imputación tradicional: MEDIANA --------------------------------------
work_med = df.copy()
mediana_age = work_med['age'].median()

print(mediana_age)

In [ ]:

work_med['age_mediana'] = work_med['age'].fillna(mediana_age)

# Guarda para contraste posterior con k-NN
age_original = df['age']          # antes (con NaN)
age_mediana  = work_med['age_mediana']  # después (mediana)

# --- Visualización y resumen ---------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Resumen numérico lado a lado
pd.DataFrame({
    'antes': age_original,
    'despues_mediana': age_mediana
}).describe().T

In [ ]:
# Histogramas comparativos
fig, axes = plt.subplots(1, 2, figsize=(10,4), sharey=True)
sns.histplot(age_original, ax=axes[0], kde=True)
axes[0].set_title('Edad – con NaN')

sns.histplot(age_mediana, ax=axes[1], kde=True, color='orange')
axes[1].set_title('Edad – imputación MEDIANA')
plt.suptitle('Antes vs. después de imputar por mediana')
plt.show()


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler

#pip install scikit-learn


# Copia de trabajo
work = df.copy()

# Ejemplo simple: imputar edad (numérico) con mediana
work['age_mediana'] = work['age'].fillna(work['age'].median())

# k‑NN Imputer en variables numéricas
num_cols = work.select_dtypes('number').columns
knn_imp = KNNImputer(n_neighbors=5)
work[num_cols] = knn_imp.fit_transform(work[num_cols])

# IterativeImputer (MICE) solo para demostración en numéricas
mice_imp = IterativeImputer(random_state=0, max_iter=10)
work[num_cols] = mice_imp.fit_transform(work[num_cols])

print('Imputación completada.')

In [ ]:
# --- Comparar variable 'age' antes y después de imputar --------------------
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Construir dos series comparables
age_original = df['age']                 # antes (con NaN)
age_imputada = work['age_mediana']       # después de imputar (o 'age' tras KNN/MICE)

# 2. Resumen numérico
comp = pd.DataFrame({
    'antes': age_original,
    'después': age_imputada
})
display(comp.describe().T)               # media, IQR, etc.

# 3. Histograma lado a lado
fig, axes = plt.subplots(1, 2, figsize=(10,4), sharey=True)
sns.histplot(age_original, ax=axes[0], kde=True)
axes[0].set_title('Antes de imputar')
sns.histplot(age_imputada, ax=axes[1], kde=True, color='orange')
axes[1].set_title('Después de imputar')
plt.suptitle('Distribución de edad\n(NaN → mediana/KNN/MICE)')
plt.show()


## 3. Preparación – Detección y tratamiento de *outliers*

In [ ]:
from scipy.stats.mstats import winsorize

# Función IQR para identificar valores atípicos en 'fare'
q1, q3 = np.percentile(work['fare'], [25, 75])
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = work[(work['fare'] < lo) | (work['fare'] > hi)]
print(f'Outliers detectados en fare: {outliers.shape[0]}')


In [ ]:

# Winsorizar
work['fare_winz'] = winsorize(work['fare'], limits=[0.01, 0.01])

# Comparar distribuciones
fig, axes = plt.subplots(1,2, figsize=(10,4))
sns.histplot(work['fare'], ax=axes[0])
axes[0].set_title('Original')
sns.histplot(work['fare_winz'], ax=axes[1])
axes[1].set_title('Winsorizado')
plt.show()

# 4.1. Análisis Exploratorio

In [ ]:
# Grafico de barra de frecuencias de 'survived'
sns.countplot(x='survived', data=work)
# Configuración de estilo
plt.title('Frecuencia de Supervivencia')
plt.xlabel('Supervivió (0=No, 1=Sí)')
plt.ylabel('Frecuencia')
plt.xticks(ticks=[0, 1], labels=['No', 'Sí'])
plt.show()

# Grafico de barra de frecuencias de 'pclass'
sns.countplot(x='pclass', data=work)
# Configuración de estilo
plt.title('Frecuencia de Supervivencia')
plt.xlabel('Clase (1, 2, 3)')
plt.ylabel('Frecuencia')
plt.xticks(ticks=[0, 1, 2], labels=['1ª Clase', '2ª Clase', '3ª Clase'])
plt.show()

# Histograma de 'fare'
sns.histplot(work['fare'], bins=30, kde=True)
plt.title('Distribución de Tarifas')
plt.xlabel('Tarifa')
plt.ylabel('Frecuencia')
plt.show()


In [ ]:
# Grafico de barras para comparar supervivencia por clase
sns.countplot(x='class', hue='survived', data=work)
# Configuración de estilo
plt.title('Supervivencia por Clase')
plt.xlabel('Clase')
plt.ylabel('Frecuencia')
plt.xticks(rotation=45)
plt.legend(title='Supervivió', loc='upper right', labels=['No', 'Sí'])
plt.show()


In [ ]:
# Grafico de boxplot para comparar edad por supervivencia
sns.boxplot(x='survived', y='age', data=work)
# Configuración de estilo
plt.title('Edad por Supervivencia')
plt.xlabel('Supervivió (0=No, 1=Sí)')
plt.ylabel('Edad')
plt.xticks(ticks=[0, 1], labels=['No', 'Sí'])
plt.show()


In [ ]:

# Grafico de boxplot para comparar Tarifa por supervivencia
sns.boxplot(x='survived', y='fare', data=work)
# Configuración de estilo
plt.title('Tarifa por Supervivencia')
plt.xlabel('Supervivió (0=No, 1=Sí)')
plt.ylabel('Tarifa')
plt.xticks(ticks=[0, 1], labels=['No', 'Sí'])
plt.show()


In [ ]:
# Grafico de boxplot para comparar tarifa por clase
sns.boxplot(x='class', y='fare', data=work)
# Configuración de estilo
plt.title('Tarifa por Clase')
plt.xlabel('Clase')
plt.ylabel('Tarifa')
plt.xticks(rotation=45)
plt.show()

# Histograma de tarifa por clase
sns.histplot(data=work, x='fare', hue='class', multiple='stack', bins=30)
plt.title('Distribución de Tarifas por Clase')
plt.xlabel('Tarifa')
plt.ylabel('Frecuencia')
plt.legend(title='Clase', loc='upper right', labels=['3ª Clase', '2ª Clase', '1ª Clase'])
plt.show()



In [ ]:

# Grafico de boxplot para comparar Tarifa por supervivencia
sns.boxplot(x='survived', y='fare', data=work)
# Configuración de estilo
plt.title('Tarifa por Supervivencia')
plt.xlabel('Supervivió (0=No, 1=Sí)')
plt.ylabel('Tarifa')
plt.xticks(ticks=[0, 1], labels=['No', 'Sí'])
plt.show()


In [ ]:

# Grafico de boxplot para comparar Tarifa por supervivencia
sns.boxplot(x='survived', y='fare', data=work)
# Configuración de estilo
plt.title('Tarifa por Supervivencia')
plt.xlabel('Supervivió (0=No, 1=Sí)')
plt.ylabel('Tarifa')
plt.xticks(ticks=[0, 1], labels=['No', 'Sí'])
plt.show()


In [ ]:

# Grafico de violinplot para comparar edad por clase
sns.violinplot(x='class', y='age', data=work)
# Configuración de estilo
plt.title('Edad por Clase')
plt.xlabel('Clase')
plt.ylabel('Edad')
plt.xticks(rotation=45)
plt.show()


In [ ]:

# Grafico de violinplot para comparar edad por supervivencia
sns.violinplot(x='survived', y='age', data=work)
# Configuración de estilo
plt.title('Edad por Supervivencia')
plt.xlabel('Supervivió (0=No, 1=Sí)')
plt.ylabel('Edad')
plt.xticks(ticks=[0, 1], labels=['No', 'Sí'])
plt.show()


## 4. Preparación – Codificación de variables categóricas

In [ ]:
# One‑hot encoding con drop_first para evitar multicolinealidad
dummies = pd.get_dummies(work['embarked'], prefix='embarked', drop_first=True)
work = pd.concat([work, dummies], axis=1)

# Target / Mean Encoding manual
mean_target = work.groupby('embarked')['survived'].mean()
work['embarked_mean_enc'] = work['embarked'].map(mean_target)

work[['embarked', 'embarked_mean_enc']].head()

## 5. Diagnóstico de multicolinealidad

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Seleccionar numéricas e independientes
features = work[['age_mediana', 'fare_winz']]
features = sm.add_constant(features)  # añadir intercepto requerido por statsmodels

# Calcular VIF
vif_df = pd.DataFrame()
vif_df['variable'] = features.columns
vif_df['VIF'] = [variance_inflation_factor(features.values, i) for i in range(features.shape[1])]
vif_df

## 6. Escalado y transformaciones

In [ ]:
from sklearn.preprocessing import MinMaxScaler, PowerTransformer

scaler_std = StandardScaler()
scaler_mm  = MinMaxScaler()
pt_yj      = PowerTransformer(method='yeo-johnson')

# Example on 'fare_winz'
work['fare_std'] = scaler_std.fit_transform(work[['fare_winz']])
work['fare_mm']  = scaler_mm.fit_transform(work[['fare_winz']])
work['fare_yj']  = pt_yj.fit_transform(work[['fare_winz']])

work[['fare_winz', 'fare_std', 'fare_mm', 'fare_yj']].head()

## 7. Ingeniería de características

In [ ]:
# Interacción simple: 'fare_yj' x 'pclass'
work['fare_pclass_int'] = work['fare_yj'] * work['pclass']

# Ver primeras filas
work[['fare_yj','pclass','fare_pclass_int']].head()

# One-hot encoding

In [ ]:
work_one_hot = pd.get_dummies(work, columns=['pclass'], drop_first=True)

work_one_hot.info()

### Preguntas de reflexión  
1. ¿Qué variable generó más *imputaciones* y cómo podría afectar al modelo?  
2. ¿Hay indicadores de que el dataset tenga sesgo de género o clase?  
3. ¿Qué técnica usarías si los outliers fueran parte fundamental del fenómeno a modelar?  
